# 任务③：LLM 辅助的轨迹清洗评估工作流

本 Notebook 演示以下闭环：

诊断 → 检索已准入记忆 → LLM 独立提出候选 → 确定性执行 → 独立搜索内部参考 → 核验 → L1 留档

LLM 负责提出参数、预期方向和依据；数值计算由确定性代码完成，候选质量由 regret 量化。确定性搜索从默认参数开始，不以 LLM 候选为搜索起点。搜索结果是同一内部目标下的参考，不是现实道路真值。

所有案例可以进入 L1 留档；只有通过核验的案例会被后续检索并进入 L2。

---
### 目录
1. 环境与数据
2. 单条轨迹完整闭环
3. 清洗和指标图
4. 离线 Mock 消融演示
5. 记忆的可解释视图
6. 读取独立的真实 ECNU 实验结果


## 1. 环境与数据

In [ ]:
import os, sys, json, warnings
from pathlib import Path
from dotenv import load_dotenv

# 本地密钥保存在被 Git 忽略的 .env；Notebook 中不写入真实 key
load_dotenv(Path(".env"), override=False)
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath("."))

# matplotlib 需要可写缓存目录
os.environ.setdefault("MPLCONFIGDIR", os.path.abspath(".mplcache"))

from traj_agent.core import traj as tm, diagnosis, params as params_mod
from traj_agent.tools.registry import ToolRegistry, attach_dataset
from traj_agent.agent.loop import TrajCleaningAgent
from traj_agent.agent import provider as prov
from traj_agent.memory.store import MemoryStore
from traj_agent.tools.playbook import vault_status

DATA = "traj_dict.json"
raw = tm.load_raw(DATA)
print(f"载入 {len(raw)} 辆车")

status = prov.provider_status()
print("\nLLM provider 配置：")
for k, v in status.items():
    print(f"  {k}: {v}")
print("\nObsidian vault：", json.dumps(vault_status(), ensure_ascii=False))


### 固定随机样本中的三种轨迹状态

下一个单元读取固定随机种子 20260920 的 200 条样本。当前诊断代码重算后，静止、混合和行驶分别为 94、48、58 条（47%、24%、29%）。这些比例只描述该样本，不能替代 11,386 辆车的全量分布。


In [ ]:
sample_manifest = json.loads(Path(
    "experiments/llm_assisted_20260920/output/random_sample_200.json"
).read_text(encoding="utf-8"))
sample = sample_manifest["vehicle_ids"]
cards = [diagnosis.diagnose(tm.traj_from_raw(v, *raw[v])) for v in sample]
sample_summary = diagnosis.dataset_regime_summary(cards)
print(json.dumps({
    "seed": sample_manifest["seed"],
    "n": len(sample),
    **sample_summary,
}, ensure_ascii=False, indent=1))


## 2. 单条轨迹完整闭环

In [ ]:
mem = MemoryStore(":memory:")      # 演示用内存库；生产换成 sqlite 文件路径
agent = TrajCleaningAgent(llm=prov.MockProvider(), memory=mem,
                          regret_threshold=0.05).attach_dataset(raw)

VEHICLE = "246"     # 11 km 的真实行驶轨迹
result = agent.run(VEHICLE)
print(f"ok={result.ok}  来源={result.proposal_source}  "
      f"工具调用={result.tool_calls}  LLM 轮次={result.llm_turns}")


### 诊断卡（这就是 LLM 看到的全部数据 —— 不含任何坐标）

In [ ]:
print(json.dumps(result.diagnosis, ensure_ascii=False, indent=1))

### trace：智能体每一步做了什么

这张表就是课堂讲评的材料 —— 能看到 LLM 调了哪些工具、
每次核验算出了什么数。

In [ ]:
for s in result.trace:
    detail = json.dumps(s.detail, ensure_ascii=False)
    print(f"[{s.index:2d}] {s.kind:6s} {s.name:22s} {detail[:110]}")

### 提议与核验结果

`regret` 是**可自动判分**的核心数字：
相对确定性搜索找到的最优解，这条建议丢掉了多少比例的可得收益。

In [ ]:
d = result.to_dict(with_trace=False)
print("LLM 提议的参数：")
print(json.dumps(d["proposal_params"], ensure_ascii=False, indent=1))
print("\n它的依据：")
print(result.proposal_raw.get("rationale", ""))
print("\n它预测的效果（会被逐一比对符号）：")
print(json.dumps(result.proposal_raw.get("expected_effect", {}), ensure_ascii=False))
print("\n实测指标（对清洗后轨迹，隔离出纯 DP 误差）：")
print(json.dumps({k: v for k, v in d["measured"].items() if k != "clean_report"},
                 ensure_ascii=False, indent=1))


In [ ]:
v = d["verification"]
print(f"提议得分   : {v['proposal_score']}")
print(f"基线得分   : {v['baseline_score']}   （物理先验默认参数）")
print(f"搜索最优   : {v['best_score']}")
print(f"regret     : {v['regret']}   口径={v['regret_basis']}")
print(f"方向准确率 : {v['direction_accuracy']}")
print(f"约束满足   : {v['constraint_ok']}   夹紧项={v['clamped_params']}")
print(f"记忆准入   : {v['admitted']}  —— {v['admit_reason']}")

## 3. 出图

In [ ]:
from traj_agent.core import clean, simplify
from traj_agent.report import figures

t_raw = tm.traj_from_raw(VEHICLE, *raw[VEHICLE])
t_clean, report = clean.denoise_trajectory(t_raw)
t_simp = simplify.simplify_trajectory(t_clean, d["proposal_params"]["dp_tolerance"]).traj

print("清洗账本：")
print(json.dumps(report.to_dict(), ensure_ascii=False, indent=1))

p1 = figures.plot_clean_overlay(t_simp, reference=t_raw, out_dir="figures",
                                filename=f"notebook_overlay_{VEHICLE}.png")
p2 = figures.plot_anomaly_scatter(t_raw, out_dir="figures",
                                  filename=f"notebook_anomaly_{VEHICLE}.png")
p3 = figures.plot_heatmap(t_simp, reference=t_raw, out_dir="figures",
                          filename=f"notebook_heatmap_{VEHICLE}.png")
print("\n输出：", p1, p2, p3, sep="\n  ")


### 质量—压缩率曲线

这张图**同时服务任务②和任务③**：
任务②看阈值扫描的边际递减形状，
任务③它就是 verifier 计算 regret 的目标函数地形图。

In [ ]:
from traj_agent.core import segment

reg = ToolRegistry(); attach_dataset(reg.ctx, raw)
h = reg.call("load_trajectory", vehicle_id=VEHICLE)["handle"]
hc = reg.call("clean_trajectory", handle=h)["handle"]
curve = reg.call("run_search", handle=hc, param="dp_tolerance", n=8,
                 reference_handle=hc)["curve"]
knee = reg.call("find_knee", curve_json=json.dumps(curve))["knee"]
p4 = figures.plot_quality_compression(curve, knee, out_dir="figures",
                                     filename=f"notebook_curve_{VEHICLE}.png")

print(f"{'容差':>8} {'点数':>6} {'压缩率':>8} {'最大偏差':>10} {'长度比':>8}")
for c in curve:
    print(f"{c['value']:8.1f} {c['n_points']:6d} {c['compression_ratio']:8.3f} "
          f"{c['max_deviation_m']:10.2f} {c['length_ratio']:8.4f}")
print("\nknee point：", json.dumps(knee, ensure_ascii=False))


In [ ]:
rows = segment.sweep_split_thresholds(t_clean, [15, 30, 60, 120],
                                        [100, 200, 400, 800, 1600])
p5 = figures.plot_sensitivity_heatmap(rows, out_dir="figures",
                                     filename=f"notebook_sensitivity_{VEHICLE}.png")
p6 = figures.plot_ablation.__doc__ and None  # 占位，消融图在第 4 节出
print("敏感性热图：", p5)

## 4. 消融实验

四种模式在结构上不同：

| 模式 | LLM | 记忆 | 搜索 |
|---|---|---|---|
| llm-only | 有 | 无 | 无 |
| search-only | 无 | 无 | 有 |
| llm+search | 有 | 无 | 有 |
| llm+memory+search | 有 | 有 | 有 |

本节仅用 MockProvider 做教学演示，不调用真实 API。search-only 必须保持零 LLM 调用。正式实验将 demo 与 holdout 分开，holdout 只读记忆并排除自身。

搜索开关会改变 regret 的口径，所以四模式不能按平均 regret 排名。跨模式使用 proposal_score - baseline_score、超过基线比例、约束违规率和成本；llm+search 与 llm+memory+search 再做同 case 配对比较。


In [ ]:
from traj_agent.verifier.verify import ablation_summary
from traj_agent.verifier.ablation import MODE_SPECS

VEHICLES = ["246", "256", "306", "209"]
cases, per_mode = [], {}
for mode, spec in MODE_SPECS.items():
    memory = MemoryStore(":memory:") if spec["use_memory"] else None
    registry = ToolRegistry()
    attach_dataset(registry.ctx, raw)
    demo_agent = TrajCleaningAgent(
        registry=registry,
        llm=prov.MockProvider(),
        memory=memory,
        mode=mode,
        **spec,
    )
    per_mode[mode] = []
    for vid in VEHICLES:
        case_result = demo_agent.run(vid)
        if not case_result.ok:
            continue
        case_dict = case_result.to_dict(with_trace=False)
        verification = case_dict["verification"]
        row = {
            "mode": mode,
            "regret": verification["regret"],
            "regret_basis": verification["regret_basis"],
            "proposal_score": verification["proposal_score"],
            "baseline_score": verification["baseline_score"],
            "n_evaluations": case_dict["llm_tool_calls"],
            "direction_accuracy": verification["direction_accuracy"],
            "admitted": verification["admitted"],
            "n_neighbors_available": case_dict["memory_prior_available"],
        }
        cases.append(row)
        per_mode[mode].append(row)
    if memory:
        memory.close()
    print(f"{mode:22s} 完成 {len(per_mode[mode])} 条")

rows = ablation_summary(cases)
print()
print(f"{'模式':22s} {'n':>3} {'相对基线':>10} {'超基线率':>9} {'LLM工具':>8}")
for row in rows:
    print(
        f"{row['mode']:22s} {row['n']:3d} "
        f"{str(row['mean_gap_to_baseline']):>10} "
        f"{str(row['baseline_beaten_rate']):>9} "
        f"{str(row['mean_evals']):>8}"
    )

p6 = figures.plot_ablation(
    rows, out_dir="figures", filename="notebook_ablation.png"
)
print("\nMock 消融图：", p6)


### 离线消融结果的解释边界

上表只验证 MockProvider 下四种模式能够按预定结构运行，并检查 search-only 的 LLM 工具调用为零。Mock 是确定性规则，不代表真实模型输出；4 条教学样本也不用于推断总体效果。跨模式结论以相对默认基线得分为主，不直接比较不同 regret 口径。


## 5. 记忆的可解释视图

记忆检索使用 12 维归一化特征与 regime 硬门控，不使用 embedding。特征均有物理含义，可以直接查看邻居及相似度。

L1 保存通过和未通过核验的案例；默认检索仅使用 admitted=1，L2 也只从已准入案例蒸馏。这样既保留失败记录用于复盘，也避免失败参数成为后续推荐。


In [ ]:
from traj_agent.memory import retrieve as retrieve_mod
from traj_agent.memory import features as feat_mod

card = diagnosis.diagnose(t_raw)
view = retrieve_mod.explain_neighbors(card, agent.memory, k=3)
print("查询轨迹的特征向量：")
print(json.dumps(view["query"]["features"], ensure_ascii=False, indent=1))
print("\n检索到的邻居：")
for nb in view["neighbors"]:
    print(f"  {nb['seg_id']:10s} 相似度={nb['similarity']:.4f} regime匹配={nb['regime_match']} "
          f"参数={nb['params']} regret={nb['regret']}")
print("\n说明：", view["note"])


In [ ]:
# L2：从已准入案例蒸馏出的参数区间
n = agent.memory.rebuild_procedural(min_samples=1)
print(f"蒸馏出 {n} 条参数区间\n")
print(retrieve_mod.region_hint_text(retrieve_mod.prior_for(card, agent.memory)))
print("\n记忆库统计：")
print(json.dumps(agent.memory.stats(), ensure_ascii=False, indent=1))

### 时间轴退化轨迹

当前 traj_dict.json 中车辆 352 有 170 个点、54 个不同时间戳，首末跨度 543 秒；正时间间隔中位数 10 秒、P95 为 20 秒。零间隔比例约 68.6%，最长连续 39 个点同一时刻，最大表观速度约 242 m/s。这里应优先判断时间戳伪影，不能把高速度点直接当成坐标漂移大量删除。


In [ ]:
r352 = agent.run("352")
d352 = r352.to_dict(with_trace=False)
print("时间轴诊断：", json.dumps(r352.diagnosis["timeline"], ensure_ascii=False))
print("observations：", json.dumps(r352.diagnosis["observations"], ensure_ascii=False, indent=1))
print("\n提议：", json.dumps(d352["proposal_params"], ensure_ascii=False))
print("依据：", r352.proposal_raw.get("rationale", ""))
print(f"\nregret={d352['verification']['regret']:.4f} 准入={d352['verification']['admitted']}")
print("理由：", d352["verification"]["admit_reason"])

### 静止轨迹：目标函数不适用

车辆 0 清洗后只有 21 点、总长 12.29 m，全是厘米级抖动。
此时任何容差都会让长度比崩掉 —— 但这不是"方案不好"，
而是**压缩率与保真度在这个尺度上没有物理含义**。

核验器对此显式标记 `applicable=False`，改按 regime 正确性准入，
而不是强行算一个负分或虚高的 regret。

In [ ]:
r0 = agent.run("0")
d0 = r0.to_dict(with_trace=False)
print("regime:", d0["regime"], " 长度相关:", json.dumps(r0.diagnosis["space"], ensure_ascii=False))
print("proposal_objective:", json.dumps(d0["proposal_objective"], ensure_ascii=False, indent=1))
print("\n核验：", json.dumps(d0["verification"], ensure_ascii=False, indent=1))

## 6. 独立真实 ECNU 实验

Notebook 开头只读取被 Git 忽略的本地环境配置。前面的课程演示显式使用 MockProvider，Restart Kernel → Run All 不会调用真实 API。

真实实验由 experiments/llm_assisted_ecnu_20260921/run_real_ablation.py 顺序执行，运行前强制检查 provider 为 OpenAICompatProvider；结果写入独立目录并支持 case 级断点恢复。下面只读取已经保存的 summary.json，不产生新费用。


In [ ]:
status = prov.provider_status()
print(json.dumps(status, ensure_ascii=False, indent=1))
print("\n本 Notebook 的所有 agent 单元均显式使用 MockProvider。")
print("真实结果只从独立实验目录读取；本单元不构造请求、不调用 API。")


In [ ]:
real_summary_path = Path(
    "experiments/llm_assisted_ecnu_20260921/summary.json"
)
if real_summary_path.exists():
    real_summary = json.loads(real_summary_path.read_text(encoding="utf-8"))
    print("ECNU model:", real_summary["experiment"]["model"])
    print("real LLM calls:", real_summary["actual_usage"]["llm_calls"])
    print("prompt/completion tokens:",
          real_summary["actual_usage"]["prompt_tokens"],
          real_summary["actual_usage"]["completion_tokens"])
    print()
    for mode, item in real_summary["mode_summary"].items():
        delta = item["score_delta"]
        print(
            f"{mode:22s} n={delta['n']:2d} "
            f"mean_delta={delta['mean']} "
            f"beaten_rate={item['baseline_beaten_rate']['mean']}"
        )
else:
    print("真实实验 summary.json 尚未生成。")


## 小结

| 问题 | 当前做法 |
|---|---|
| LLM 候选是否合理 | 物理先验约束参数空间，LLM 独立提出候选与方向 |
| 如何量化候选质量 | 确定性执行候选；独立搜索建立内部参考；Verifier 计算 regret |
| 如何跨模式比较 | 使用相对默认基线得分、超基线率、可靠性和成本 |
| 如何避免记忆污染 | 全部案例进 L1；仅 admitted=1 可检索并蒸馏到 L2 |
| 如何避免信息泄漏 | demo 与 holdout 分离，holdout 只读并排除自身 |
| 如何控制真实 API 费用 | Notebook 固定 Mock；真实实验独立、顺序、可恢复 |

当前内部指标不能替代道路真值。OSM 或人工标注可作为后续独立评价，但不在本轮主实验中改写 primary objective。
